In [41]:
# SCRAPPING MUBI AWARDS PAGE

In [42]:
# USING PLAYWRIGHT METHOD TO AVOID BLOCKING FROM WEBSITE

In [40]:
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
import os

CHROMIUM_WS_ENDPOINT = os.getenv("PLAYWRIGHT_WS_ENDPOINT")

# Url to scrap MUBI festivals
BASE_URL = "https://mubi.com/"
FESTIVAL_URL = f"{BASE_URL}fr/awards-and-festivals/{{festival}}?page={{page_num}}&year={{year}}"
FILM_URL = f"{BASE_URL}{{film_link}}"

AWARDS_MUBI_LIST = [
  "oscars",
  "cesars",
  "baftas",
  "golden-globes"
]

FESTIVALS_MUBI_LIST = [
  "cannes",
  "venice",
  "berlinale"
]

DEFAULT_FILTERS = {
  "festival": "cesars",
  "page_num": "1",
  "year": "2024",
}

MUBI_FESTIVAL_CLASSES = {
  "movies": "ul.css-liprzy", # ul - css-liprzy e3o0xsn0
  "movie": "li.css-1ncbtb3", # li - css-1ncbtb3 ejkdwq71
  "nominations": "div.css-gyp8mm", # div - css-gyp8mm eiz03ik4 OR div - css-ko5a18 eiz03ik1
  "title": "h3.css-1hr6q83", # h3 - css-1hr6q83 e1fxv8uz1
  "director": "span.css-1vg6q84", # span - css-1vg6q84 e1slvksg0
  "country": "span.css-ahepiu", # span - css-ahepiu epl1xvv1
  "year": "p.css-1t8l3k8", # span inside this div - css-7pofbu epl1xvv0 (no specific class for this one)
  "link": "a.css-122y91a" # css-122y91a e8qvidc2
}

# Scraping MUBI website method
async def scrape_mubi_festivals_async(filters = DEFAULT_FILTERS):
  generated_url = FESTIVAL_URL.format(**filters)

  async with async_playwright() as p:
    # Simulate a real browser to get the full page content
    browser = await p.chromium.connect_over_cdp(CHROMIUM_WS_ENDPOINT)  # Connect to remote Chromium
    context = await browser.new_context(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
    page = await context.new_page()
    
    await page.goto(generated_url) #, wait_until="networkidle")
    await page.wait_for_timeout(2000)
    await page.mouse.wheel(0, 1000)
    await page.wait_for_timeout(2000)

    html = await page.content()

    # Extract from page content the movies data
    soup = BeautifulSoup(html, "html.parser")
    movie_elements = soup.select(MUBI_FESTIVAL_CLASSES["movie"])
    movies = []
    for movie in movie_elements:
        title = movie.select_one(MUBI_FESTIVAL_CLASSES["title"])
        director = movie.select_one(MUBI_FESTIVAL_CLASSES["director"])
        country = movie.select_one(MUBI_FESTIVAL_CLASSES["country"])
        nominations = movie.select_one(MUBI_FESTIVAL_CLASSES["nominations"])
        link = movie.select_one(MUBI_FESTIVAL_CLASSES["link"])

        movie_data = {
            "title": title.get_text(strip=True) if title else None,
            "director": director.get_text(strip=True) if director else None,
            "country": country.get_text(strip=True) if country else None,
            "nominations": nominations.get_text(strip=True) if nominations else None,
            "link": link["href"] if link else None
        }

        movies.append(movie_data)

    await browser.close()
    return movies


# Run the scraping method
await scrape_mubi_festivals_async()

[{'title': 'PERFECT DAYS',
  'director': 'Wim Wenders',
  'country': 'Japon',
  'nominations': '1 nomination',
  'link': '/fr/fr/films/perfect-days'},
 {'title': 'LES FEUILLES MORTES',
  'director': 'Aki Kaurismäki',
  'country': 'Finlande',
  'nominations': '1 nomination',
  'link': '/fr/fr/films/fallen-leaves-2023'},
 {'title': 'YANNICK',
  'director': 'Quentin Dupieux',
  'country': 'France',
  'nominations': '2 nominations',
  'link': '/fr/fr/films/yannick'},
 {'title': "ANATOMIE D'UNE CHUTE",
  'director': 'Justine Triet',
  'country': 'France',
  'nominations': 'Best Film & 6 autres',
  'link': '/fr/fr/films/anatomy-of-a-fall'},
 {'title': 'LA PASSION DE DODIN BOUFFANT',
  'director': 'Trần Anh Hùng',
  'country': 'France',
  'nominations': '3 nominations',
  'link': '/fr/fr/films/pot-au-feu-2023'},
 {'title': 'BOLERO',
  'director': 'Nans Laborde-Jourdàa',
  'country': 'France',
  'nominations': '1 nomination',
  'link': '/fr/fr/films/bolero-2023'},
 {'title': 'NOTRE CORPS',
  '

In [45]:
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
import os

CHROMIUM_WS_ENDPOINT = os.getenv("PLAYWRIGHT_WS_ENDPOINT")

# Url to scrap MUBI festivals
BASE_URL = "https://mubi.com"
FILM_URL = f"{BASE_URL}{{film_link}}/awards"

DEFAULT_FILM_FILTERS = {
  "film_link": "/fr/fr/films/the-substance"
}

MUBI_FILM_CLASSES = {
  "award": "div.css-epmkt5",
  "festival": "a.css-pgwez",
  "reward": "div.css-16kkjs",
}

# Scraping MUBI website method
async def scrap_mubi_film_awards(filters = DEFAULT_FILM_FILTERS):
  generated_url = FILM_URL.format(**filters)

  async with async_playwright() as p:
    # Simulate a real browser to get the full page content
    browser = await p.chromium.connect_over_cdp(CHROMIUM_WS_ENDPOINT)  # Connect to remote Chromium
    context = await browser.new_context(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
    page = await context.new_page()
    
    await page.goto(generated_url) #, wait_until="networkidle")
    await page.wait_for_timeout(2000)
    await page.mouse.wheel(0, 1000)
    await page.wait_for_timeout(2000)

    html = await page.content()
    with open("mubi_film.html", "w") as file:
        file.write(html)

    # Extract from page content the movies data
    soup = BeautifulSoup(html, "html.parser")
    award_elements = soup.select(MUBI_FILM_CLASSES["award"])
    awards = []
    for award in award_elements:
        festival = award.select_one(MUBI_FILM_CLASSES["festival"])
        reward = award.select_one(MUBI_FILM_CLASSES["reward"])

        award_data = {
            "festival": festival.get_text(strip=True) if festival else None,
            "reward": reward.get_text(strip=True) if reward else None,
        }

        awards.append(award_data)

    await browser.close()
    return awards


# Run the scraping method
await scrap_mubi_film_awards()


[{'festival': 'Festival de Cannes',
  'reward': '2024 | Lauréat : Prix du scénario'},
 {'festival': 'Academy Awards',
  'reward': '2025 | Nommé: Meilleur scénario original'},
 {'festival': 'Academy Awards',
  'reward': '2025 | Lauréat : Best Makeup and Hairstyling'},
 {'festival': 'Academy Awards', 'reward': '2025 | Nommé: Meilleur film'},
 {'festival': 'Academy Awards',
  'reward': '2025 | Nommé: Meilleure interprétation féminine'},
 {'festival': 'Academy Awards',
  'reward': '2025 | Nommé: Meilleur réalisateur'},
 {'festival': 'Independent Spirit Awards',
  'reward': '2025 | Nommé: Best Feature'},
 {'festival': 'Independent Spirit Awards',
  'reward': '2025 | Nommé: Best Lead Performance'},
 {'festival': 'Toronto International Film Festival',
  'reward': '2024 | Lauréat : Prix du public'},
 {'festival': 'San Sebastián International Film Festival',
  'reward': '2024 | Sélection Officielle'},
 {'festival': "Indiewire Critics' Poll", 'reward': '2024 | Nommé: Best Film'},
 {'festival': "

In [47]:
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
import os

CHROMIUM_WS_ENDPOINT = os.getenv("PLAYWRIGHT_WS_ENDPOINT")

# Url to scrap MUBI festivals
BASE_URL = "https://mubi.com"
ALL_FESTIVALS_URL = f"{BASE_URL}/fr/awards-and-festivals?type={{festival_or_award}}&page={{page_num}}"

DEFAULT_FESTIVALS_FILTERS = {
  "page_num": "1",
  "festival_or_award": "festival",
}

MUBI_FESTIVALS_CLASSES = {
  "festival": "div.css-te72z8",
  "festival_link": "a.css-13zcxcg",
  "festival_name": "div.css-1o91brm"
}

# Scraping MUBI website method
async def scrap_mubi_film_festivals(filters = DEFAULT_FESTIVALS_FILTERS):
  generated_url = ALL_FESTIVALS_URL.format(**filters)

  async with async_playwright() as p:
    # Simulate a real browser to get the full page content
    browser = await p.chromium.connect_over_cdp(CHROMIUM_WS_ENDPOINT)  # Connect to remote Chromium
    context = await browser.new_context(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
    page = await context.new_page()
    
    await page.goto(generated_url) #, wait_until="networkidle")
    await page.wait_for_timeout(2000)
    await page.mouse.wheel(0, 1000)
    await page.wait_for_timeout(2000)

    html = await page.content()
    with open("mubi_film.html", "w") as file:
        file.write(html)

    # Extract from page content the movies data
    soup = BeautifulSoup(html, "html.parser")
    festival_elements = soup.select(MUBI_FESTIVALS_CLASSES["festival"])
    festivals = []
    for festival in festival_elements:
        festival_link = festival.select_one(MUBI_FESTIVALS_CLASSES["festival_link"])
        festival_name = festival.select_one(MUBI_FESTIVALS_CLASSES["festival_name"])

        festival_data = {
            "festival_link": festival.get_text(strip=True) if festival_link else None,
            "festival_name": reward.get_text(strip=True) if festival_name else None,
        }

        festivals.append(festival_data)

    await browser.close()
    return festivals


# Run the scraping method
await scrap_mubi_film_festivals()


[]